# Example 7.1 / 1: Solve a Production Model in Whole Batches

Companion notebook for **7.1 Integer Variables** in the AMPL lab Documentation, section "Solve a Production Model in Whole Batches". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a Production Model in Whole Batches

Suppose products must be made in whole batches: bands earn 25 per batch and take 2 hours, coils
earn 32 and take 3 hours, and 17 hours are available.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products made in whole batches
param profit {PROD};              # profit per batch
param hours {PROD} > 0;           # hours needed per batch
param available >= 0;             # total hours available

var Make {PROD} integer >= 0;     # number of batches of each product

maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];  # profit of all batches
# batch hours used fit within the hours available
subject to Capacity: sum {p in PROD} hours[p] * Make[p] <= available;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 32}
ampl.param["hours"] = {"bands": 2, "coils": 3}
ampl.param["available"] = 17

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["TotalProfit"].value())
print(ampl.var["Make"].to_dict())

HiGHS 1.14.0: optimal solution; objective 207
0 simplex iterations
1 branching nodes
solved 207.0
{'bands': 7, 'coils': 1}


### Pure AMPL: data.dat and model.run

`ampl.solve_result` is the status, `ampl.obj["TotalProfit"].value()` the objective and
`to_dict()` the batch counts: 7 batches of bands and 1 of coils use all 17 hours.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param profit := bands 25 coils 32;
param hours := bands 2 coils 3;
param available := 17;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display TotalProfit, Make;

HiGHS 1.14.0: optimal solution; objective 207
0 simplex iterations
1 branching nodes
TotalProfit = 207

Make [*] :=
bands  7
coils  1
;



### Another way in amplpy

The **continuous relaxation** is the same model with `integer` dropped, so every point between
integer values is allowed. AMPL's `relax_integrality` option solves it without editing the model
(see Fixing and Relaxing Variables (AMPL lab documentation, section 6.4)). With the same `model.mod` and `data.py`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 32}
ampl.param["hours"] = {"bands": 2, "coils": 3}
ampl.param["available"] = 17

In [11]:
# run.py
ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs", verbose=False)
print("relaxation:", ampl.obj["TotalProfit"].value(), ampl.var["Make"].to_dict())
ampl.option["relax_integrality"] = 0
ampl.solve(solver="highs", verbose=False)
print("integer:   ", ampl.obj["TotalProfit"].value(), ampl.var["Make"].to_dict())

relaxation: 212.5 {'bands': 8.5, 'coils': 0}
integer:    207.0 {'bands': 7, 'coils': 1}


### Pure AMPL: data.dat and model.run

The relaxation makes 8.5 batches of bands, the product with the best profit per hour, and earns
212.5. No integer plan can earn more than that, so the relaxation is an upper bound for this
maximization; the integer optimum, 207, switches to a different mix. How solvers use this bound
is the subject of Relaxations, Bounds and Optimality Gaps (AMPL lab documentation, section 7.1.1).

The same in pure AMPL:

In [12]:
%%writefile data.dat
set PROD := bands coils;
param profit := bands 25 coils 32;
param hours := bands 2 coils 3;
param available := 17;

Overwriting data.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
option relax_integrality 1;
solve;
display TotalProfit, Make;
option relax_integrality 0;
solve;
display TotalProfit, Make;

HiGHS 1.14.0: optimal solution; objective 212.5
0 simplex iterations
0 barrier iterations
TotalProfit = 212.5

Make [*] :=
bands  8.5
coils  0
;

HiGHS 1.14.0: optimal solution; objective 207
0 simplex iterations
1 branching nodes
TotalProfit = 207

Make [*] :=
bands  7
coils  1
;

